# Train De Novo Model

After completing the CLIP pre-training stage, the next step is to train the actual De Novo sequencing model.

Below is a detailed introduction to the parameters in the denovo configuration file:

## clip_checkpoint_path (Crucial)

It specifies the exact path to the optimal `.ckpt` weight file generated during your Stage 1 (CLIP) training. The model will load the pre-trained spectrum encoder from this checkpoint.

## aug (Data Augmentation)
enabled: In the denovo stage, this is typically set to false.

## tokenizer

The tokenizer settings should generally be identical to the ones you used in the CLIP stage to ensure vocabulary and tensor dimensions align perfectly.

## model (Model Architecture)

Defines the Transformer parameters. These should generally match your CLIP configuration to successfully load the pre-trained weights.

## loss

label_smoothing: A regularization technique (e.g., 0.05).

## optimizer & scheduler

optimizer:
-   `lr`: The base learning rate (3.0e-4).
-   `weight_decay`: L2 penalty (1.0e-4).

scheduler:
-   `enabled`: Whether to enable the scheduler.
-   `warmup_steps`: The number of warmup steps. This parameter is highly flexible:

    You can provide a float less than 1.0 (e.g., 0.1), and the system will automatically multiply it by the total training steps to calculate the dynamic warmup step percentage.

    You can also provide an integer greater than 1 (e.g., 9500 as in the example) to force a specific, absolute number of warmup steps.

-   `n_cycles`: The number of cosine cycles (restarts) over the entire training process. If set to 1 (Standard), the learning rate follows a single, smooth cosine decay curve down to zero after the warmup. If set to > 1 (e.g., 3), the learning rate will "restart" to a peak value periodically, which can help the model escape local minima.
-   `lr_decay_factor`: The decay multiplier applied to the peak learning rate at each restart. For example, if this is 5.0 and n_cycles is 3, the peak learning rate of the 2nd cycle will be $1/5$ (20%) of the base learning rate, and the 3rd cycle will be $1/25$ (4%).
    -   ⚠️ Important Note: This parameter strictly controls the peak decay between cycles. Therefore, if n_cycles is set to 1, lr_decay_factor becomes an inactive dummy variable, and its value (whether 1.0 or 10.0) will have no effect on the final learning rate curve.

## trainer

-   `evaluate_metric_name`: Changed to "total_accuracy".
-   `task_name`: Set to "denovo".
-   `devices` / `distributed`: Hardware configuration (e.g., devices: [0, 1], distributed: "ddp" for multi-GPU training).
-   `summarywriter_folder` / `model_save_folder`: The directories for TensorBoard logs and saving the final De Novo checkpoints.

## data

-   `train_path` / `val_path`: Paths to the HDF5 files.
-   `train_batch_size` / `val_batch_size`: The batch size for each stage (train/val).

    You may notice the batch sizes here (192) with a RTX4090 GPU.
-   `n_workers`: The number of dataloader worker processes.

The `test_path` and `test_batch_size` can be left unset. We don't use them.

In [1]:
from ruamel.yaml import YAML

# replace the data path with your own path

yaml = YAML()
yaml.preserve_quotes = True
yaml.indent(mapping=2, sequence=4, offset=2)

config = """
aug:
  enabled: false

clip_checkpoint_path: ""

tokenizer:
  spectrum:
    n_top_peaks: 300
    min_mz: 50.0
    max_mz: 4500.0
    min_intensity: 0.01
    remove_precursor_tol: 2.0
  
  peptide:
    reverse: true
    residues: "massivekb"

model:
  spectrum:
    hidden_size: 512
    n_head: 8
    n_layers: 9
    dropout: 0.18
    dim_feedforward: 1024
    activation: "gelu"
  
  peptide:
    n_vocab: 29
    hidden_size: 512
    n_head: 8
    n_layers: 9
    dropout: 0.18
    dim_feedforward: 1024
    activation: "gelu"

loss:
  label_smoothing: 0.05

optimizer:
  lr: 3.0e-4
  weight_decay: 1.0e-4

scheduler:
  enabled: true
  n_cycles: 1
  warmup_steps: 37000
  lr_decay_factor: 10.0

trainer:
  is_max: true
  evaluate_metric_name: "total_accuracy"
  task_name: "denovo"
  random_seed: 1256
  save_top_k: 3
  max_epochs: 10
  devices: [0, 1]
  grad_norm_clip: 1.5
  validation_steps: 50000
  show_progress_bar: true
  grad_scaler_enable: true
  distributed: "ddp"
  gradient_accumulation_steps: 1
  summarywriter_folder: "./outputs/tb"
  model_save_folder: "./outputs/checkpoints/denovo"

data:
  train_path: ""
  val_path: ""
  test_path: ""
  train_batch_size: 192
  val_batch_size: 192
  test_batch_size: 256
  n_workers: 16
"""

config = yaml.load(config)
yaml.dump(config, open("/data2/xp/RocNovo-Lightning/outputs/denovo.yaml", "w"))

In [ ]:
import sys
sys.path.append("..")

from rocnovo.module.denovo import train as train_denovo

train_denovo("/data2/xp/RocNovo-Lightning/outputs/denovo.yaml")